# Simple Regression Project — Diabetes Dataset

Follow the cells from top to bottom. Each code cell performs one small step. Target: predict a continuous diabetes progression score.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Load the dataset
Keep `diabetes_regression.csv` in the same folder as this notebook.

In [ ]:
df = pd.read_csv("diabetes_regression.csv")
print("Rows and columns:", df.shape)
df.head()

## 3. Understand the data

In [ ]:
df.info()

In [ ]:
print("Missing values in each column:")
print(df.isnull().sum())

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
df.describe()

## 4. Data cleaning
This cell removes exact duplicate rows. It fills missing numeric feature values with the median. The target is not imputed; rows with a missing target are removed. The imputer inside each model pipeline is also fitted only on training folds.

In [ ]:
df = df.drop_duplicates()
df = df.dropna(subset=["target"])

X = df.drop(columns=["target"])
y = df["target"]

# Fill any missing input values with each column's median
imputer = SimpleImputer(strategy="median")
X_clean = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

print("Shape after cleaning:", X_clean.shape)
print("Missing values after cleaning:", X_clean.isnull().sum().sum())

## 5. Visualize the data

In [ ]:
df.hist(figsize=(12, 8))
plt.tight_layout()
plt.show()

## 6. Split data into training and testing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y, test_size=0.20, random_state=42
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 7. Create models
Scaling and median imputation are placed inside pipelines for models that need scaling. Trees do not need scaling, but imputation is included.

In [ ]:
models = {
    "Linear Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),
    "KNN Regressor": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", KNeighborsRegressor())
    ]),
    "SVR": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", SVR())
    ]),
    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeRegressor(random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(random_state=42))
    ])
}

## 8. Compare models using 5-fold cross-validation

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
results = []

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="r2")
    results.append([name, scores.mean()])

cv_results = pd.DataFrame(results, columns=["Model", "Average CV R2"])
cv_results.sort_values("Average CV R2", ascending=False)

## 9. Tune each model
GridSearchCV tries multiple settings and selects the best using training folds.

In [ ]:
parameter_grids = {
    "Linear Regression": {"model__fit_intercept": [True, False]},
    "KNN Regressor": {
        "model__n_neighbors": [3, 5, 7, 9],
        "model__weights": ["uniform", "distance"]
    },
    "SVR": {
        "model__C": [0.1, 1, 10],
        "model__kernel": ["linear", "rbf"]
    },
    "Decision Tree": {
        "model__max_depth": [None, 3, 5, 10],
        "model__min_samples_leaf": [1, 2, 4]
    },
    "Random Forest": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [None, 5, 10]
    }
}

In [ ]:
best_models = {}
tuning_results = []

for name in models:
    search = GridSearchCV(
        models[name],
        parameter_grids[name],
        cv=cv,
        scoring="r2",
        n_jobs=-1
    )
    search.fit(X_train, y_train)
    best_models[name] = search.best_estimator_
    tuning_results.append([name, search.best_score_, search.best_params_])

pd.DataFrame(
    tuning_results,
    columns=["Model", "Best CV R2", "Best Parameters"]
).sort_values("Best CV R2", ascending=False)

## 10. Evaluate tuned models on test data

In [ ]:
test_results = []

for name, model in best_models.items():
    predictions = model.predict(X_test)
    mae = mean_absolute_error(y_test, predictions)
    rmse = mean_squared_error(y_test, predictions) ** 0.5
    r2 = r2_score(y_test, predictions)
    test_results.append([name, mae, rmse, r2])

test_results = pd.DataFrame(
    test_results, columns=["Model", "MAE", "RMSE", "R2"]
).sort_values("R2", ascending=False)

test_results

In [ ]:
best_name = test_results.iloc[0]["Model"]
print("Highest test R2 model:", best_name)

## 11. Plot actual vs predicted values

In [ ]:
best_model = best_models[best_name]
predictions = best_model.predict(X_test)

plt.scatter(y_test, predictions)
plt.xlabel("Actual value")
plt.ylabel("Predicted value")
plt.title("Actual vs Predicted")
plt.show()

## 12. Conclusion
Use the values shown in `test_results` to write your conclusion. Higher R2 and lower MAE/RMSE indicate better test-set performance. Do not report results until you run the notebook.